# Customer Churn Prediction Pipeline

- train model

In [0]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Normalizer
from pyspark.ml.classification import LogisticRegression, NaiveBayes, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import roc_curve, auc

spark = (
    SparkSession.builder
    .appName("ChurnAnalysis")
    .config("spark.sql.pushdown.enabled", "true")
    .getOrCreate()
)

WORK_PATH = os.getcwd()
VOLUME_PATH = "/Volumes/workspace/default/customer_data"
RAW_DIR = f"{VOLUME_PATH}/raw"
PARTITIONED_DIR = f"{VOLUME_PATH}/partitioned"
GRAPHS_DIR = f"{WORK_PATH}/graphs"
ML_EXPERIMENT = f"{WORK_PATH}/customer_churn_experiments"
CHURN_CSV = f"{RAW_DIR}/churn.csv"

# Create the volume directories if they don't already exist
# (mkdirs also creates intermediate/parent directories as needed)
for _dir in (RAW_DIR, PARTITIONED_DIR):
    dbutils.fs.mkdirs(_dir)

print("Directories ensured:", RAW_DIR, "|", PARTITIONED_DIR, "|", GRAPHS_DIR)

In [0]:
churn_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(CHURN_CSV)
)

print("Rows:", churn_csv.count())
churn_csv.printSchema()
display(churn_csv.limit(10))


In [0]:
numerical_cols = ["CreditScore", "Age", "Balance", "EstimatedSalary", "NumOfProducts", "Tenure"]
churn_csv.select(numerical_cols).summary().display()

In [0]:
num_duplicates = churn_csv.count() - churn_csv.distinct().count()
print(f"There are {num_duplicates} duplicates in this dataset.")

In [0]:
float_cols = {c for c, dtype in churn_csv.dtypes if dtype in ("float", "double")}
cols = churn_csv.columns

# Compute counts horizontally first
agg_exprs = [
    F.count(
        F.when(F.col(c).isNull() | (F.isnan(c) if c in float_cols else F.lit(False)), c)
    ).alias(c)
    for c in cols
]

# Unpivot into a clean vertical table
stack_expr = ", ".join([f"'{c}', `{c}`" for c in cols])

churn_csv.select(agg_exprs) \
    .select(F.expr(f"stack({len(cols)}, {stack_expr}) as (column_name, null_count)")) \
    .show(len(cols), truncate=False)

In [0]:
# Collect only the numerical columns to the driver as Row objects, then plot each in the grid
numerical_rows = churn_csv.select(numerical_cols).collect()

ncols = 2
nrows = (len(numerical_cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(10, 3.5 * nrows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    values = [r[col] for r in numerical_rows]
    axes[i].hist(values, bins=30, color="#2b5c8f", alpha=0.7)
    axes[i].set_title(col)
    axes[i].grid(axis="y", linestyle="--", alpha=0.3)

# Hide any unused grid slots
for j in range(len(numerical_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
TARGET_SAMPLE_SIZE = 10000

total_count = churn_csv.count()
counts_per_class = churn_csv.groupBy("Exited").count().collect()

sample_ratio = min(1.0, TARGET_SAMPLE_SIZE / total_count)
fractions = {row["Exited"]: sample_ratio for row in counts_per_class}

# Stratified sample using native expressions (rand + comparison) instead of
# sampleBy, which compiles to a Scala UDF filter that Photon cannot accelerate.
rand_col = "__sample_rand"
keep_row = F.lit(False)
for stratum, fraction in fractions.items():
    keep_row = keep_row | ((F.col("Exited") == stratum) & (F.col(rand_col) < fraction))

some_churn = (
    churn_csv
    .withColumn(rand_col, F.rand(seed=69))
    .where(keep_row)
    .drop(rand_col)
)

In [0]:
some_churn.groupBy("Exited").agg(
    F.round(F.mean("Age"), 1).alias("avgAge"),
    F.round(F.mean("IsActiveMember") * 100, 1).alias("PctActiveMembers"),
    F.round(F.mean("HasCrCard") * 100, 1).alias("PctCrCardHolders"),
    F.round(F.mean("NumOfProducts"), 2).alias("avgNumOfProducts"),
    F.round(F.mean("Balance"), 2).alias("avgBalance"),
    F.round(F.mean("EstimatedSalary"), 2).alias("avgEstimatedSalary"),
    F.round(F.mean("Tenure"), 1).alias("avgTenure")
).show()

some_churn.groupBy("Geography", "Gender").agg(
    F.count("*").alias("TotalCustomers"),
    F.round(F.mean("Balance"),1).alias("avgBalance"),
    F.round(F.mean("CreditScore")).alias("avgScore").cast('int'),
    F.sum("Exited").alias("ChurnedCount")
).sort(F.desc('avgBalance')).show()


In [0]:

# 1. Exclude ID / identifier columns
excluded_cols = {"RowNumber", "CustomerId", "Surname"}
candidate_cols = [c for c in churn_csv.columns if c not in excluded_cols]

# 2. Encode categorical string columns (Geography, Gender) to numeric for correlation
df_encoded = churn_csv
feature_cols = []

for c, dtype in churn_csv.select(candidate_cols).dtypes:
    if dtype == "string":
        indexed_col = f"{c}_idx"
        indexer = StringIndexer(inputCol=c, outputCol=indexed_col)
        df_encoded = indexer.fit(df_encoded).transform(df_encoded)
        feature_cols.append(indexed_col)
    else:
        feature_cols.append(c)

# 3. Assemble features into a vector and compute Pearson correlation in Spark
assembler = VectorAssembler(inputCols=feature_cols, outputCol="corr_features")
vector_df = assembler.transform(df_encoded).select("corr_features")

corr_matrix = Correlation.corr(vector_df, "corr_features", method="pearson").collect()[0][0].toArray()

# 4. Clean display labels (strip '_idx')
display_labels = [c.replace("_idx", "") for c in feature_cols]
n = len(display_labels)

# 5. Plot Heatmap using pure Matplotlib
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

# Set tick labels
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(display_labels, rotation=45, ha="right")
ax.set_yticklabels(display_labels)

# Add correlation coefficients inside cells
for i in range(n):
    for j in range(n):
        val = corr_matrix[i, j]
        text_color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=text_color, fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson Correlation")
ax.set_title("Feature Correlation Heatmap", fontsize=13, pad=12)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# 1. Bucket ages < 30 into '20' (18-30), >= 80 into '80' (80+), and the rest in 10-year intervals
age_bucket_df = (
    some_churn
    .withColumn(
        "age_bin_start",
        F.when(F.col("Age") < 30, 20)
         .when(F.col("Age") >= 80, 80)
         .otherwise((F.floor(F.col("Age") / 10) * 10).cast("int"))
    )
    .groupBy("age_bin_start")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("Exited") == 1, 1).otherwise(0)).alias("churned_count"),
        F.sum(F.when(F.col("Exited") == 0, 1).otherwise(0)).alias("retained_count"),
    )
    .withColumn("churn_pct", (F.col("churned_count") / F.col("total")) * 100)
    .withColumn("retained_pct", (F.col("retained_count") / F.col("total")) * 100)
    .orderBy("age_bin_start")
)

# 2. Collect as standard Python Row objects
rows = age_bucket_df.collect()

# Format edge labels as '18-30' and '80+'
def format_age_label(bin_start):
    if bin_start == 20:
        return "18-30"
    elif bin_start == 80:
        return "80+"
    return f"{bin_start}-{bin_start + 10}"

x_labels = [format_age_label(r["age_bin_start"]) for r in rows]
churn_pct = [r["churn_pct"] for r in rows]
retained_pct = [r["retained_pct"] for r in rows]
x_pos = range(len(x_labels))

# 3. 100% Stacked Bar Chart with Matplotlib
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.bar(
    x_pos,
    churn_pct,
    label="Churned (1)",
    color="red",
    alpha=0.5,
    width=0.65
)
ax.bar(
    x_pos,
    retained_pct,
    bottom=churn_pct,
    label="Retained (0)",
    color="green",
    alpha=0.5,
    width=0.65
)

# Formatting
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, rotation=45)
ax.set_xlabel("Age Group")
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 100)
ax.set_title("Retention vs. Churn Rate by Age Group (100% Stacked)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
# Save the figure directly into the graphs volume (direct write works on this compute)
plt.savefig(f"{GRAPHS_DIR}/churn_by_age_group.png")
plt.show()

In [0]:
some_churn.createOrReplaceTempView("churn")

In [0]:
import matplotlib.pyplot as plt
from matplotlib import ticker

sql_query = """
    SELECT 
        Balance,
        IsActiveMember,
        ROUND(
            AVG(Exited) OVER (
                PARTITION BY IsActiveMember
                ORDER BY Balance
                ROWS BETWEEN 500 PRECEDING AND 500 FOLLOWING
            ) * 100.0, 
            2
        ) AS rolling_churn_rate
    FROM churn
    WHERE Balance > 0
    ORDER BY Balance
"""

# 1. Collect as a Python list of Row objects (no pandas)
rows = spark.sql(sql_query).collect()

# 2. Plot using list comprehensions to filter and extract columns
fig, ax = plt.subplots(figsize=(10, 5))

for active_status, label, color in [
    (0, "Inactive Member (0)", "#d95f02"),
    (1, "Active Member (1)", "#2b5c8f")
]:
    balances = [r["Balance"] * 1e-3 for r in rows if r["IsActiveMember"] == active_status]
    rates = [r["rolling_churn_rate"] for r in rows if r["IsActiveMember"] == active_status]
    
    ax.plot(
        balances, 
        rates, 
        label=label, 
        color=color, 
        linewidth=2
    )

ax.set_ylim(10, 40)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{x:.0f}k"))
ax.set_xlabel("Balance")
ax.set_ylabel("Churn Probability (%)")
ax.set_title("Continuous Churn Rate by Balance (Split by Member Activity)")
ax.legend(loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/churn_rate_by_balance.png")
plt.show()

In [0]:
# Compute churn rate directly in Spark
prod_churn = (
    some_churn.groupBy("NumOfProducts")
    .agg(
        F.count("*").alias("Total"),
        (F.mean("Exited") * 100).alias("ChurnRate")
    )
    .orderBy("NumOfProducts")
    .toPandas()
)

fig, ax1 = plt.subplots(figsize=(8, 4.5))

# Bar chart: Total Customers
color = "#2b5c8f"
ax1.set_xlabel("Number of Products")
ax1.set_ylabel("Total Customers", color=color)
bars = ax1.bar(prod_churn["NumOfProducts"], prod_churn["Total"], color=color, alpha=0.6, width=0.5)
ax1.tick_params(axis="y", labelcolor=color)

# Line chart: Churn Rate (%)
ax2 = ax1.twinx()
color = "#d95f02"
ax2.set_ylabel("Churn Rate (%)", color=color)
ax2.plot(prod_churn["NumOfProducts"], prod_churn["ChurnRate"], color=color, marker="o", linewidth=2.5)
ax2.tick_params(axis="y", labelcolor=color)

plt.title("Customer Volume vs. Churn Rate by Number of Products")
plt.tight_layout()
plt.savefig(f"{GRAPHS_DIR}/churn_by_products.png")
plt.show()

In [0]:
spark.sql("""
    SELECT 
        Geography, 
        COUNT(*) AS TotalCustomers, 
        ROUND(SUM(Exited)/TotalCustomers*100, 1) AS PercentLost
    FROM churn
    GROUP BY Geography
    ORDER BY PercentLost DESC
""").show()

spark.sql("""
    SELECT 
        Gender,
        COUNT(*) AS TotalCustomers,
        ROUND(SUM(Exited)/TotalCustomers*100, 1) AS PercentLost
    FROM churn
    GROUP BY Gender
    ORDER BY PercentLost DESC;
""").show()

## ML branch — preprocessing → stratified split → partitioned Delta → model input

The EDA branch above works on `some_churn` (stratified sample, unpartitioned — samples stay small by design).
The ML branch below works on the **full** dataset:

1. Preprocess (drop IDs, encode categoricals, assemble ALL predictors into one `features` vector) and keep **only** `features`, the label `Exited`, and `Geography` — the raw columns are not needed downstream
2. Split train/test **stratified on y** (`Exited`): the top 80% of each class goes to train, so class balance is identical on both sides
3. Write once as Delta with `partitionBy("Geography", "is_train")` — **country first, then train/test**
4. Read back — model training consumes `train_df` / `test_df`

Every country appears in both the train and test directories: the split is the *second* partition level,
so no whole country ends up on one side of the split.

In [0]:
# ML branch: FULL dataset (not the EDA sample some_churn). Reuses churn_csv from the raw read.
ID_COLS = ["RowNumber", "CustomerId", "Surname"]

ml_base = churn_csv.drop(*ID_COLS)

# Encode categoricals as numeric features. Geography plays both roles:
#   - kept as the string column that physically partitions the Delta table (directory key)
#   - encoded to Geography_idx so the model still gets country signal
gender_indexer = StringIndexer(inputCol="Gender", outputCol="Gender_idx").fit(ml_base)
geo_indexer = StringIndexer(inputCol="Geography", outputCol="Geography_idx").fit(ml_base)

ml_features = geo_indexer.transform(gender_indexer.transform(ml_base))

# Assemble every predictor into ONE features vector. Excluded: Exited (the label y),
# Geography (the partition column — the model gets its signal via Geography_idx),
# and the raw Gender string (superseded by Gender_idx).
FEATURE_EXCLUDE = {"Exited", "Geography", "Gender"}
feature_cols = [c for c in ml_features.columns if c not in FEATURE_EXCLUDE]

# Keep ONLY what the model needs: the vectorized features, the label y, and the
# partition key. The split and the partitioned write below operate on this compact
# frame — not on the wide dataset with all its raw columns.
model_input = (
    VectorAssembler(inputCols=feature_cols, outputCol="features")
    .transform(ml_features)
    .select("Geography", "Exited", "features")
)

print("feature_cols:", feature_cols)
model_input.printSchema()

In [0]:
TRAIN_FRACTION = 0.8

# Stratified split on y (Exited): rank rows WITHIN each class and send the top
# TRAIN_FRACTION of each class to train, so the churn/retain proportions carry over
# exactly to both sides — important on imbalanced data like this.
stratified_split = (
    model_input
    # Materialize rand first so the window can order by it as a plain column
    .withColumn("__rand", F.rand(seed=69))
    .withColumn("class_rank", F.row_number().over(Window.partitionBy("Exited").orderBy("__rand")))
    .withColumn("class_size", F.count(F.lit(1)).over(Window.partitionBy("Exited")))
    .withColumn("is_train", F.col("class_rank") <= F.ceil(F.col("class_size") * TRAIN_FRACTION))
    .drop("class_rank", "class_size", "__rand")
)

# Partition FIRST by country, THEN by train/test. Directory layout:
#   PARTITIONED_DIR/Geography=France/is_train=true/...
#   PARTITIONED_DIR/Geography=France/is_train=false/...
# Every country appears on both sides — the split is the second partition level,
# so no whole country lands on one side of the train/test split.
# mode("overwrite") keeps notebook re-runs idempotent. overwriteSchema is needed
# here because this write also changes the table's schema/partitioning versus
# any earlier layout (e.g. Geography-only partitioning) already at that path.
(
    stratified_split.write
    .format("delta")
    .partitionBy("Geography", "is_train")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(PARTITIONED_DIR)
)
print(f"Wrote Delta table partitioned by Geography, then is_train, to {PARTITIONED_DIR}")

In [0]:
# Downstream ML reads from the partitioned table — never re-derives from the raw CSV.
ml_data = spark.read.format("delta").load(PARTITIONED_DIR)

# Sanity check: partition layout — one Geography=France/ ... directory per country
print("Partitions:", [r["Geography"] for r in ml_data.select("Geography").distinct().collect()])

# Train/test from the is_train partition level: both splits span all countries
train_df = ml_data.where(F.col("is_train"))
test_df = ml_data.where(~F.col("is_train"))

print("total:", ml_data.count(), "| train:", train_df.count(), "| test:", test_df.count())
display(ml_data.groupBy("Geography", "is_train").count().orderBy("Geography", "is_train"))

In [0]:
import mlflow

# Create the experiment if it does not exist, and make it the default
# destination for every mlflow.start_run() in this notebook.
if mlflow.get_experiment_by_name(ML_EXPERIMENT) is None:
    mlflow.create_experiment(ML_EXPERIMENT)

exp = mlflow.set_experiment(ML_EXPERIMENT)
print(f"Logging runs to experiment '{ML_EXPERIMENT}' (id: {exp.experiment_id})")

In [0]:
# Spark ML consumes ONE DataFrame holding BOTH the features vector and the label —
# an sklearn-style X/y split (separate frames) cannot be fed to .fit(). The
# partitioned Delta table already carries exactly those columns, so alias compact
# frames for training/evaluation.
# .cache() is deliberate: LogisticRegression reads its input repeatedly
# (fit + summary + transform), and without it every access re-scans the Delta files.
train_ml = train_df.select("features", "Exited").cache()
test_ml = test_df.select("features", "Exited").cache()

train_rows = train_ml.count()
test_rows = test_ml.count()
train_churn = train_ml.select(F.mean("Exited")).first()[0]
test_churn = test_ml.select(F.mean("Exited")).first()[0]

print(f"train rows: {train_rows} | test rows: {test_rows}")
print(f"churn rate train: {train_churn:.3f} | test: {test_churn:.3f} (stratified split keeps them equal)")

In [0]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaled_features",
                        withMean=True, withStd=True)
scaler_model = scaler.fit(train_ml)
scaled_train = scaler_model.transform(train_ml).cache()
scaled_test = scaler_model.transform(test_ml).cache()  # kept for Step 2 evaluation

def f1_macro(preds, label_col="Exited"):
    """Macro-averaged F1: the unweighted mean of the per-class F1 scores."""
    cm = (preds.groupBy(F.col(label_col).alias("label"), F.col("prediction").alias("pred"))
               .count()
               .collect())
    counts = {(float(r["label"]), float(r["pred"])): r["count"] for r in cm}

    def _f1(c):
        tp = counts.get((float(c),     float(c)),     0)
        fp = counts.get((float(1 - c),  float(c)),     0)  # predicted c, true is other
        fn = counts.get((float(c),     float(1 - c)), 0)  # true c, predicted other
        return 2 * tp / (2 * tp + fp + fn) if tp > 0 else 0.0

    return (_f1(0) + _f1(1)) / 2.0

In [0]:
def train_and_log_model(
    estimator,
    train_df,
    run_name: str,
    model_family: str,
    label_col: str = "Exited",
    extra_params: dict = None
):
    """
    Fits a PySpark ML estimator on train_df, evaluates training AUC-ROC and F1-macro
    via the DataFrame evaluator API (serverless-safe), and logs everything to MLflow.
    """
    with mlflow.start_run(run_name=run_name) as run:
        fitted_model = estimator.fit(train_df)

        # Evaluate via pure DataFrame API (avoids blocked serverless checkpoint mounts)
        auc_evaluator = BinaryClassificationEvaluator(
            labelCol=label_col,
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        )
        train_preds = fitted_model.transform(train_df)
        train_auc = auc_evaluator.evaluate(train_preds)
        train_f1 = f1_macro(train_preds)

        print(f"[{run_name}] Training AUC-ROC:   {train_auc:.4f}")
        print(f"[{run_name}] Training F1 (macro): {train_f1:.4f}")

        mlflow.set_tag("model_family", model_family)

        # Safely extract feature count (models like GBT/RF use numFeatures; Pipelines may not)
        n_features = getattr(fitted_model, "numFeatures", None)
        if n_features is None:
            first_row = train_df.select(estimator.getFeaturesCol()).first()
            n_features = len(first_row[0]) if first_row else None

        params = {
            "n_features": n_features,
            "train_rows": train_df.count(),
        }
        if extra_params:
            params.update(extra_params)

        mlflow.log_params(params)
        mlflow.log_metric("train_auc_roc", train_auc)
        mlflow.log_metric("train_f1_macro", train_f1)

        run_id = run.info.run_id

    print(f"MLflow run recorded: {run_id}\n")
    return fitted_model, run_id

In [0]:
lr = LogisticRegression(featuresCol="scaled_features", labelCol="Exited")
rf = RandomForestClassifier(featuresCol="features", labelCol="Exited", numTrees=300, maxDepth=4)
nb = NaiveBayes(featuresCol="features", labelCol="Exited", modelType="multinomial")

In [0]:
import datetime as dt
models = {'Logistic Regression':lr, 'Random Forest': rf, 'Naive Bayes': nb}
results = {}

for nr, (name, model) in enumerate(models.items(), start=1):
    run_name = f"run{nr}_{name}_{dt.date.today()}"
    fitted_model, run_id = train_and_log_model(
        estimator=model,
        train_df=scaled_train,
        run_name=run_name,
        model_family=name
    )
    results[name] = {
        "model": fitted_model,
        "run_id": run_id
    }